In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.applications.inception_v3 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
 
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)

In [ ]:
DATASET_ROOT = "/kaggle/input/datasets/aviarvasu/ds1-split/DS1_split"          
TRAIN_DIR    = os.path.join(DATASET_ROOT, "/kaggle/input/datasets/aviarvasu/ds1-split/DS1_split/TRAIN")
TEST_DIR     = os.path.join(DATASET_ROOT, "/kaggle/input/datasets/aviarvasu/ds1-split/DS1_split/TEST")
 
IMG_SIZE     = (299, 299)           
BATCH_SIZE   = 32
SEED         = 42
 
# Phase-1  
P1_EPOCHS    = 20
P1_LR        = 1e-3
 
# Phase-2  
P2_EPOCHS    = 20
P2_LR        = 1e-4
UNFREEZE_FROM = 249                
 
MODEL_SAVE_PATH = "waste_classifier.keras"
CLASS_NAMES  = ["N", "O", "R"]    

In [ ]:
print("\nBuilding data pipelines …")
 
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    zoom_range=0.20,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest",
    validation_split=0.15,          #
)
 
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
 
train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    seed=SEED,
    shuffle=True,
)
 
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    seed=SEED,
    shuffle=False,
)
 
test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)
 
NUM_CLASSES = len(train_gen.class_indices)
print(f" Class indices : {train_gen.class_indices}")
print(f"   Train batches : {len(train_gen)}  |  Val batches : {len(val_gen)}  |  Test batches : {len(test_gen)}")
 

In [ ]:
def build_model(num_classes: int) -> tf.keras.Model:
    base = InceptionV3(
        weights="imagenet",
        include_top=False,
        input_shape=(*IMG_SIZE, 3),
    )
    base.trainable = False          
 
    inputs = tf.keras.Input(shape=(*IMG_SIZE, 3))
    x = base(inputs, training=False)
 
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation="relu",
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu",
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
 
    return tf.keras.Model(inputs, outputs), base
 
 
model, base_model = build_model(NUM_CLASSES)
model.summary()

In [ ]:
# phase 1 — Training new head 
print("\nPhase 1 — Training classification head (base frozen) …\n")
 
model.compile(
    optimizer=optimizers.Adam(learning_rate=P1_LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
 
p1_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    ModelCheckpoint(
        "best_phase1.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]
 
history_p1 = model.fit(
    train_gen,
    epochs=P1_EPOCHS,
    validation_data=val_gen,
    callbacks=p1_callbacks,
    verbose=1,
)
 

In [ ]:
# phase 2 — Fine-tune top layers of base
print(f"\nPhase 2 — Unfreezing InceptionV3 from layer {UNFREEZE_FROM} …\n")
 
base_model.trainable = True
for layer in base_model.layers[:UNFREEZE_FROM]:
    layer.trainable = False
 
model.compile(
    optimizer=optimizers.SGD(learning_rate=P2_LR, momentum=0.9, nesterov=True),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)
 
p2_callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=4,
        min_lr=1e-7,
        verbose=1,
    ),
    ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
]
 
history_p2 = model.fit(
    train_gen,
    epochs=P2_EPOCHS,
    validation_data=val_gen,
    callbacks=p2_callbacks,
    verbose=1,
)
 
print(f"\nBest model saved → {MODEL_SAVE_PATH}")

In [ ]:
#  plotting training history
def plot_history(h1, h2, save_path="training_curves.png"):
    acc1  = h1.history["accuracy"]
    vacc1 = h1.history["val_accuracy"]
    loss1 = h1.history["loss"]
    vloss1 = h1.history["val_loss"]
 
    acc2  = h2.history["accuracy"]
    vacc2 = h2.history["val_accuracy"]
    loss2 = h2.history["loss"]
    vloss2 = h2.history["val_loss"]
 
    all_acc   = acc1  + acc2
    all_vacc  = vacc1 + vacc2
    all_loss  = loss1 + loss2
    all_vloss = vloss1 + vloss2
    p2_start  = len(acc1)
 
    epochs_range = range(1, len(all_acc) + 1)
 
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("InceptionV3 — Waste Classifier Training", fontsize=15, fontweight="bold")
 
    for ax, metric, v_metric, ylabel in zip(
        axes,
        [all_acc, all_loss],
        [all_vacc, all_vloss],
        ["Accuracy", "Loss"],
    ):
        ax.plot(epochs_range, metric,   label=f"Train {ylabel}", linewidth=2)
        ax.plot(epochs_range, v_metric, label=f"Val {ylabel}",   linewidth=2, linestyle="--")
        ax.axvline(p2_start, color="red", linestyle=":", linewidth=1.5, label="Phase-2 starts")
        ax.set_xlabel("Epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{ylabel} over epochs")
        ax.legend()
        ax.grid(alpha=0.3)
 
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"📊  Training curves saved → {save_path}")
 
plot_history(history_p1, history_p2)

In [ ]:
#evaluation on test set
print("\n🔍  Evaluating on test set …")
 
best_model = tf.keras.models.load_model(MODEL_SAVE_PATH)
 
test_loss, test_acc = best_model.evaluate(test_gen, verbose=1)
print(f"\n  Test Loss     : {test_loss:.4f}")
print(f"  Test Accuracy : {test_acc * 100:.2f}%")

In [ ]:
#predictions and metrics
test_gen.reset()
y_pred_probs = best_model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes

labels = [test_gen.class_indices[c] for c in sorted(test_gen.class_indices, key=test_gen.class_indices.get)]
label_names = sorted(test_gen.class_indices, key=test_gen.class_indices.get)  # ['N','O','R']
 

In [ ]:
#confusion matrix
def plot_confusion_matrix(y_true, y_pred, class_names, save_path="confusion_matrix.png"):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle("Confusion Matrix — Waste Classifier", fontsize=14, fontweight="bold")
 
    for ax, data, title, fmt in zip(
        axes,
        [cm, cm_norm],
        ["Raw Counts", "Row-Normalised"],
        ["d", ".2f"],
    ):
        sns.heatmap(
            data,
            annot=True,
            fmt=fmt,
            cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names,
            linewidths=0.5,
            ax=ax,
            cbar=True,
        )
        ax.set_xlabel("Predicted Label", fontsize=11)
        ax.set_ylabel("True Label", fontsize=11)
        ax.set_title(title, fontsize=12)
 
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"📊  Confusion matrix saved → {save_path}")
 
plot_confusion_matrix(y_true, y_pred, label_names)
 
 
print("\n" + "=" * 60)
print("  PER-CLASS EVALUATION REPORT")
print("=" * 60)
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))
 
precision = precision_score(y_true, y_pred, average=None)
recall    = recall_score   (y_true, y_pred, average=None)
f1        = f1_score       (y_true, y_pred, average=None)
 
print(f"{'Class':<12} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("-" * 45)
for name, p, r, f in zip(label_names, precision, recall, f1):
    print(f"{name:<12} {p:>10.4f} {r:>10.4f} {f:>10.4f}")
 
print("-" * 45)
print(f"{'Macro avg':<12} {precision.mean():>10.4f} {recall.mean():>10.4f} {f1.mean():>10.4f}")
print("=" * 60)
 